# Notebook 02 — Baseline EDA (No-Fault Scenarios)

**Purpose:** Characterise the no-fault (`faultLocation = 0`) baseline across all three test runs. This is the empirical foundation every later notebook builds on — understanding what *healthy* looks like before we try to recognise a fault.

**Goals:**
- Load and parse all 12 baseline CSVs (4 per run × 3 runs).
- Aggregate per-fixture statistics for every telemetry family (COMM, RATES, EXCOMM, CORR).
- Plot key metrics along the ordered fixture axis 1001 → 1090 for each run.
- Quantify natural fixture-to-fixture variation and cross-run drift.
- Identify naturally noisy fixtures that need per-fixture z-score normalisation in notebook 04.
- Assess which metrics have smooth spatial profiles — best change-point candidates.
- Save `outputs/baseline_fixture_features.csv`, `outputs/baseline_stats_by_run_fixture.csv`, plots.

**Label-isolation rule:** `faultLocation` / `faultResistance` are used only to filter baseline files. They never enter any feature or prediction path.

## 1. Imports and Configuration

In [ ]:
import os
import csv
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

matplotlib.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

In [ ]:
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_ROOT    = PROJECT_ROOT / 'EFD_T0001-T0003_Data'
OUTPUTS_DIR  = PROJECT_ROOT / 'outputs'
PLOTS_DIR    = OUTPUTS_DIR / 'plots' / '02_baseline'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Canonical schemas
COMM_COLS   = ['timestamp','message_type','remote_id',
               'response_rate','num_attempts','num_failures',
               'ds_avg_repeater_depth','us_avg_repeater_depth',
               'ds_current_repeater_depth','us_current_repeater_depth',
               'avg_receive_signal_strength','incorrect_count']
RATES_COLS  = ['timestamp','message_type','remote_id',
               'dsRawRate0','dsRawRate1','dsRawRate2','dsRawRate3',
               'dsRawRate4','dsRawRate5','dsRawRate6','dsRawRate7',
               'usRawRate0','usRawRate1','usRawRate2','usRawRate3',
               'usRawRate4','usRawRate5','usRawRate6','usRawRate7']
EXCOMM_COLS = ['timestamp','message_type','remote_id',
               'peakInputLevel','overflowStatus','berCount',
               'phaseNoise_dB','rxCrcFailCount']
CORR_COLS   = ['timestamp','message_type','remote_id',
               'peakCorrLevel_dB','peakUncorrLevel_dB',
               'ftryCorrTrigCount','userCorrTrigCount',
               'corrFired','ftryCorrFired']
SCHEMA_MAP  = {'COMM': COMM_COLS, 'RATES': RATES_COLS,
               'EXCOMM': EXCOMM_COLS, 'CORR': CORR_COLS}

FIXTURE_IDS = list(range(1001, 1091))
RUN_PALETTE = {'EFD_T0001': '#1f77b4', 'EFD_T0002': '#ff7f0e', 'EFD_T0003': '#2ca02c'}

print('Project root:', PROJECT_ROOT)
print('Plots dir   :', PLOTS_DIR)

## 2. Loader

In [ ]:
def parse_csv(file_path):
    """Parse one scenario CSV into four typed DataFrames."""
    raw = {k: [] for k in SCHEMA_MAP}
    with open(file_path, 'r', newline='') as fh:
        reader = csv.reader(fh)
        for row in reader:
            row = [c.strip() for c in row]
            if len(row) < 3:
                continue
            mtype = row[1]
            if mtype not in SCHEMA_MAP:
                continue
            if len(row) != len(SCHEMA_MAP[mtype]):
                continue
            raw[mtype].append(row)

    dfs = {}
    for mtype, cols in SCHEMA_MAP.items():
        if raw[mtype]:
            df = pd.DataFrame(raw[mtype], columns=cols)
            df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
            df['remote_id'] = pd.to_numeric(df['remote_id'], errors='coerce').astype('Int64')
            for col in cols[3:]:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            df = pd.DataFrame(columns=cols)
        dfs[mtype] = df
    return dfs

print('Loader ready.')

## 3. Load Baseline Metadata and Parse All Baseline Files

In [ ]:
metadata_df   = pd.read_csv(OUTPUTS_DIR / 'metadata.csv')
baseline_meta = metadata_df[metadata_df['faultLocation'] == 0].copy().reset_index(drop=True)

print(f'Total baseline files: {len(baseline_meta)}')
display(baseline_meta.groupby('run_id').size().rename('count').to_frame())

In [ ]:
baseline_parsed = []

for _, row in baseline_meta.iterrows():
    dfs = parse_csv(row['file_path'])
    baseline_parsed.append({'run_id': row['run_id'], 'file_name': row['file_name'], **dfs})
    print(f"  {row['run_id']}  {row['file_name']}  "
          f"COMM={len(dfs['COMM'])} RATES={len(dfs['RATES'])} "
          f"EXCOMM={len(dfs['EXCOMM'])} CORR={len(dfs['CORR'])}")

print(f'\nLoaded {len(baseline_parsed)} baseline scenarios.')

## 4. Per-Fixture Feature Aggregation

Collapse each 15-minute scenario to one summary row per fixture per family, then merge all families.

In [ ]:
DEPTH_WEIGHTS = np.arange(8)

def agg_comm(df):
    if df.empty:
        return pd.DataFrame({'remote_id': FIXTURE_IDS})
    df = df.dropna(subset=['remote_id'])
    g = df.groupby('remote_id')
    out = pd.DataFrame({
        'remote_id'                 : list(g.groups.keys()),
        'comm_response_rate_mean'   : g['response_rate'].mean().values,
        'comm_response_rate_min'    : g['response_rate'].min().values,
        'comm_response_rate_std'    : g['response_rate'].std().fillna(0).values,
        'comm_num_failures_sum'     : g['num_failures'].sum().values,
        'comm_num_attempts_sum'     : g['num_attempts'].sum().values,
        'comm_ds_avg_depth_mean'    : g['ds_avg_repeater_depth'].mean().values,
        'comm_us_avg_depth_mean'    : g['us_avg_repeater_depth'].mean().values,
        'comm_signal_strength_mean' : g['avg_receive_signal_strength'].mean().values,
        'comm_incorrect_count_sum'  : g['incorrect_count'].sum().values,
    })
    out['comm_failure_rate'] = (out['comm_num_failures_sum'] /
                                out['comm_num_attempts_sum'].replace(0, np.nan)).fillna(0)
    return out


def agg_rates(df):
    if df.empty:
        return pd.DataFrame({'remote_id': FIXTURE_IDS})
    df = df.dropna(subset=['remote_id']).copy()
    ds_cols = [f'dsRawRate{k}' for k in range(8)]
    us_cols = [f'usRawRate{k}' for k in range(8)]
    ds_sum = df[ds_cols].sum(axis=1).replace(0, np.nan)
    us_sum = df[us_cols].sum(axis=1).replace(0, np.nan)
    ds_hi  = df[[f'dsRawRate{k}' for k in range(1, 8)]].sum(axis=1)
    us_hi  = df[[f'usRawRate{k}' for k in range(1, 8)]].sum(axis=1)
    df['r_higher_ds'] = (ds_hi / ds_sum).fillna(0)
    df['r_higher_us'] = (us_hi / us_sum).fillna(0)
    df['r_wavg_ds']   = ((df[ds_cols].values * DEPTH_WEIGHTS).sum(axis=1) / ds_sum).fillna(0)
    df['r_wavg_us']   = ((df[us_cols].values * DEPTH_WEIGHTS).sum(axis=1) / us_sum).fillna(0)
    df['r_asym']      = df['r_wavg_ds'] - df['r_wavg_us']
    g = df.groupby('remote_id')
    out = pd.DataFrame({
        'remote_id'                   : list(g.groups.keys()),
        'rates_dsRawRate0_mean'       : g['dsRawRate0'].mean().values,
        'rates_usRawRate0_mean'       : g['usRawRate0'].mean().values,
        'rates_higher_depth_ds_mean'  : g['r_higher_ds'].mean().values,
        'rates_higher_depth_us_mean'  : g['r_higher_us'].mean().values,
        'rates_wavg_depth_ds_mean'    : g['r_wavg_ds'].mean().values,
        'rates_wavg_depth_us_mean'    : g['r_wavg_us'].mean().values,
        'rates_ds_us_asymmetry_mean'  : g['r_asym'].mean().values,
    })
    return out


def agg_excomm(df):
    if df.empty:
        return pd.DataFrame({'remote_id': FIXTURE_IDS})
    df = df.dropna(subset=['remote_id']).copy()
    df['overflow_flag'] = (df['overflowStatus'] != 0).astype(int)
    g = df.groupby('remote_id')
    out = pd.DataFrame({
        'remote_id'               : list(g.groups.keys()),
        'ex_peakInputLevel_mean'  : g['peakInputLevel'].mean().values,
        'ex_peakInputLevel_std'   : g['peakInputLevel'].std().fillna(0).values,
        'ex_peakInputLevel_min'   : g['peakInputLevel'].min().values,
        'ex_berCount_sum'         : g['berCount'].sum().values,
        'ex_berCount_mean'        : g['berCount'].mean().values,
        'ex_phaseNoise_dB_mean'   : g['phaseNoise_dB'].mean().values,
        'ex_phaseNoise_dB_std'    : g['phaseNoise_dB'].std().fillna(0).values,
        'ex_rxCrcFailCount_sum'   : g['rxCrcFailCount'].sum().values,
        'ex_overflow_nonzero'     : g['overflow_flag'].sum().values,
    })
    return out


def agg_corr(df):
    if df.empty:
        return pd.DataFrame({'remote_id': FIXTURE_IDS})
    df = df.dropna(subset=['remote_id'])
    g = df.groupby('remote_id')
    out = pd.DataFrame({
        'remote_id'                 : list(g.groups.keys()),
        'corr_peakCorrLevel_mean'   : g['peakCorrLevel_dB'].mean().values,
        'corr_peakCorrLevel_std'    : g['peakCorrLevel_dB'].std().fillna(0).values,
        'corr_peakUncorrLevel_mean' : g['peakUncorrLevel_dB'].mean().values,
        'corr_peakUncorrLevel_std'  : g['peakUncorrLevel_dB'].std().fillna(0).values,
        'corr_corrFired_rate'       : g['corrFired'].mean().values,
        'corr_ftryCorrFired_rate'   : g['ftryCorrFired'].mean().values,
    })
    out['corr_snr_proxy'] = out['corr_peakCorrLevel_mean'] - out['corr_peakUncorrLevel_mean']
    return out


def aggregate_scenario(entry):
    base = pd.DataFrame({'remote_id': FIXTURE_IDS})
    for fn, key in [(agg_comm,'COMM'),(agg_rates,'RATES'),(agg_excomm,'EXCOMM'),(agg_corr,'CORR')]:
        base = base.merge(fn(entry[key]), on='remote_id', how='left')
    base['run_id']    = entry['run_id']
    base['file_name'] = entry['file_name']
    return base

print('Aggregation functions defined.')

In [ ]:
agg_records = [aggregate_scenario(e) for e in baseline_parsed]
agg_all     = pd.concat(agg_records, ignore_index=True)
agg_all     = agg_all.sort_values(['run_id','file_name','remote_id']).reset_index(drop=True)

FEAT_COLS = [c for c in agg_all.columns if c not in ('run_id','file_name','remote_id')]

print(f'Aggregation table: {agg_all.shape}  ({len(FEAT_COLS)} feature columns)')
display(agg_all.head(4))

## 5. Per-Run × Per-Fixture Baseline Statistics

Pool the 4 baseline files within each run to get mean/std/min/max per fixture.

In [ ]:
baseline_stats = (
    agg_all
    .groupby(['run_id', 'remote_id'])[FEAT_COLS]
    .agg(['mean', 'std', 'min', 'max'])
    .reset_index()
)
baseline_stats.columns = [
    '_'.join(c).rstrip('_') if (isinstance(c, tuple) and c[1]) else (c[0] if isinstance(c, tuple) else c)
    for c in baseline_stats.columns
]
std_cols = [c for c in baseline_stats.columns if c.endswith('_std')]
baseline_stats[std_cols] = baseline_stats[std_cols].fillna(0).clip(lower=1e-9)

global_stats = (
    agg_all
    .groupby('remote_id')[FEAT_COLS]
    .agg(['mean', 'std', 'min', 'max'])
    .reset_index()
)
global_stats.columns = [
    '_'.join(c).rstrip('_') if (isinstance(c, tuple) and c[1]) else (c[0] if isinstance(c, tuple) else c)
    for c in global_stats.columns
]

print(f'Per-run stats : {baseline_stats.shape}')
print(f'Global stats  : {global_stats.shape}')
display(baseline_stats.head(3))

## 6. Plot Helper

In [ ]:
def line_by_run(col, title='', ylabel='', fname=None, shade_col=None):
    """Per-run mean fixture profile on the ordered fixture axis."""
    fig, ax = plt.subplots(figsize=(14, 4))
    for run, color in RUN_PALETTE.items():
        sub = baseline_stats[baseline_stats['run_id'] == run].sort_values('remote_id')
        if col not in sub.columns:
            continue
        x, y = sub['remote_id'].values, sub[col].values
        ax.plot(x, y, color=color, label=run, linewidth=1.5)
        if shade_col and shade_col in sub.columns:
            s = sub[shade_col].fillna(0).values
            ax.fill_between(x, y - s, y + s, color=color, alpha=0.12)
    ax.set_xlabel('Fixture ID  (1001 -> 1090)')
    ax.set_ylabel(ylabel or col)
    ax.set_title(title or f'Baseline {col}')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    plt.tight_layout()
    if fname:
        fig.savefig(PLOTS_DIR / fname, bbox_inches='tight')
    plt.show()

print('Plot helper ready.')

## 7. EDA — COMM Family

In [ ]:
line_by_run('comm_response_rate_mean_mean',
            title='Baseline COMM - Response Rate per Fixture (shaded +-1 std)',
            ylabel='Response Rate (%)', fname='comm_response_rate.png',
            shade_col='comm_response_rate_mean_std')

In [ ]:
line_by_run('comm_failure_rate_mean',
            title='Baseline COMM - Failure Rate per Fixture',
            ylabel='Failure Rate (failures / attempts)', fname='comm_failure_rate.png',
            shade_col='comm_failure_rate_std')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, col, direction in [
    (axes[0], 'comm_ds_avg_depth_mean_mean', 'Downstream'),
    (axes[1], 'comm_us_avg_depth_mean_mean', 'Upstream'),
]:
    for run, color in RUN_PALETTE.items():
        sub = baseline_stats[baseline_stats['run_id'] == run].sort_values('remote_id')
        if col in sub.columns:
            ax.plot(sub['remote_id'], sub[col], color=color, label=run, linewidth=1.4)
    ax.set_title(f'COMM - {direction} Avg Repeater Depth (baseline)')
    ax.set_xlabel('Fixture ID'); ax.set_ylabel('Avg Depth')
    ax.legend(fontsize=8); ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'comm_repeater_depth.png', bbox_inches='tight')
plt.show()

## 8. EDA — RATES Family

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, col, direction in [
    (axes[0], 'rates_dsRawRate0_mean_mean', 'Downstream'),
    (axes[1], 'rates_usRawRate0_mean_mean', 'Upstream'),
]:
    for run, color in RUN_PALETTE.items():
        sub = baseline_stats[baseline_stats['run_id'] == run].sort_values('remote_id')
        if col in sub.columns:
            ax.plot(sub['remote_id'], sub[col], color=color, label=run, linewidth=1.4)
    ax.set_title(f'RATES - {direction} Depth-0 (Direct) Rate (baseline)')
    ax.set_xlabel('Fixture ID'); ax.set_ylabel('Rate at Depth 0 (0-1)')
    ax.legend(fontsize=8); ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'rates_depth0.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, col, direction in [
    (axes[0], 'rates_wavg_depth_ds_mean_mean', 'Downstream'),
    (axes[1], 'rates_wavg_depth_us_mean_mean', 'Upstream'),
]:
    for run, color in RUN_PALETTE.items():
        sub = baseline_stats[baseline_stats['run_id'] == run].sort_values('remote_id')
        if col in sub.columns:
            ax.plot(sub['remote_id'], sub[col], color=color, label=run, linewidth=1.4)
    ax.set_title(f'RATES - {direction} Weighted Avg Depth (baseline)')
    ax.set_xlabel('Fixture ID'); ax.set_ylabel('Weighted Avg Depth')
    ax.legend(fontsize=8); ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'rates_wavg_depth.png', bbox_inches='tight')
plt.show()

## 9. EDA — EXCOMM Family

In [ ]:
line_by_run('ex_peakInputLevel_mean_mean',
            title='Baseline EXCOMM - Peak Input Level per Fixture (shaded +-1 std)',
            ylabel='Peak Input Level (raw)', fname='excomm_peakInputLevel.png',
            shade_col='ex_peakInputLevel_mean_std')

In [ ]:
line_by_run('ex_phaseNoise_dB_mean_mean',
            title='Baseline EXCOMM - Phase Noise (dB) per Fixture (shaded +-1 std)',
            ylabel='Phase Noise (dB)', fname='excomm_phaseNoise.png',
            shade_col='ex_phaseNoise_dB_mean_std')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, col, label in [
    (axes[0], 'ex_berCount_sum_mean',      'BER Count (sum)'),
    (axes[1], 'ex_rxCrcFailCount_sum_mean', 'CRC Fail Count (sum)'),
]:
    for run, color in RUN_PALETTE.items():
        sub = baseline_stats[baseline_stats['run_id'] == run].sort_values('remote_id')
        if col in sub.columns:
            ax.plot(sub['remote_id'], sub[col], color=color, label=run, linewidth=1.4)
    ax.set_title(f'EXCOMM - {label} (baseline)')
    ax.set_xlabel('Fixture ID'); ax.set_ylabel(label)
    ax.legend(fontsize=8); ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'excomm_error_counts.png', bbox_inches='tight')
plt.show()

## 10. EDA — CORR Family

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, col, label in [
    (axes[0], 'corr_peakCorrLevel_mean_mean',   'Peak Corr Level (dB)'),
    (axes[1], 'corr_peakUncorrLevel_mean_mean',  'Peak Uncorr Level (dB)'),
]:
    for run, color in RUN_PALETTE.items():
        sub = baseline_stats[baseline_stats['run_id'] == run].sort_values('remote_id')
        if col in sub.columns:
            ax.plot(sub['remote_id'], sub[col], color=color, label=run, linewidth=1.4)
    ax.set_title(f'CORR - {label} (baseline)')
    ax.set_xlabel('Fixture ID'); ax.set_ylabel(label)
    ax.legend(fontsize=8); ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'corr_peak_levels.png', bbox_inches='tight')
plt.show()

In [ ]:
line_by_run('corr_snr_proxy_mean',
            title='Baseline CORR - SNR Proxy (CorrLevel - UncorrLevel) per Fixture',
            ylabel='SNR Proxy (dB)', fname='corr_snr_proxy.png',
            shade_col='corr_snr_proxy_std')

## 11. Spatial Heatmap — All Key Metrics vs Fixtures

In [ ]:
KEY_METRICS = [
    'comm_response_rate_mean', 'comm_failure_rate',
    'comm_ds_avg_depth_mean',  'rates_dsRawRate0_mean',
    'rates_usRawRate0_mean',   'rates_higher_depth_ds_mean',
    'rates_wavg_depth_ds_mean','ex_peakInputLevel_mean',
    'ex_phaseNoise_dB_mean',   'ex_berCount_sum',
    'corr_snr_proxy',          'corr_peakCorrLevel_mean',
    'corr_peakUncorrLevel_mean',
]
available = [m for m in KEY_METRICS if m in agg_all.columns]

hm_profile = (
    agg_all.groupby('remote_id')[available].mean().sort_index()
)

hm_norm = hm_profile.T.copy().astype(float)
for row_label in hm_norm.index:
    vals = hm_norm.loc[row_label].values
    rng  = np.nanmax(vals) - np.nanmin(vals)
    if rng > 0:
        hm_norm.loc[row_label] = (vals - np.nanmin(vals)) / rng

fig, ax = plt.subplots(figsize=(18, 5))
im = ax.imshow(hm_norm.values, aspect='auto', cmap='viridis', vmin=0, vmax=1)
x_pos = [i for i, fid in enumerate(FIXTURE_IDS) if fid % 10 == 1]
ax.set_xticks(x_pos)
ax.set_xticklabels([str(FIXTURE_IDS[i]) for i in x_pos], fontsize=8)
ax.set_yticks(range(len(hm_norm.index)))
ax.set_yticklabels(list(hm_norm.index), fontsize=8)
ax.set_xlabel('Fixture ID (1001 -> 1090)')
ax.set_title('No-Fault Baseline Spatial Profile - Key Metrics\n(each row normalised 0-1; bright = high value)')
plt.colorbar(im, ax=ax, fraction=0.01, label='Normalised value')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'baseline_spatial_heatmap.png', bbox_inches='tight')
plt.show()

## 12. Fixture-to-Fixture Variation Analysis

How much does each key metric vary across the 90 fixtures under healthy baseline? Wide natural variation requires per-fixture z-score normalisation before fault scoring.

In [ ]:
fixture_var = (
    agg_all.groupby('remote_id')[available].std().reset_index().sort_values('remote_id')
)

print('Fixture variability (std across baseline files) - circuit-level summary:')
display(fixture_var[available].describe().round(5))

# Heatmap
hv_data = fixture_var.set_index('remote_id')[available].T.copy().astype(float)
for row_label in hv_data.index:
    v = hv_data.loc[row_label].values
    rng = np.nanmax(v) - np.nanmin(v)
    if rng > 0:
        hv_data.loc[row_label] = (v - np.nanmin(v)) / rng

fig, ax = plt.subplots(figsize=(18, 5))
im2 = ax.imshow(hv_data.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(x_pos)
ax.set_xticklabels([str(FIXTURE_IDS[i]) for i in x_pos], fontsize=8)
ax.set_yticks(range(len(hv_data.index)))
ax.set_yticklabels(list(hv_data.index), fontsize=8)
ax.set_xlabel('Fixture ID (1001 -> 1090)')
ax.set_title('Natural Fixture Variability (normalised std)\nBright = naturally noisy -> needs per-fixture z-score')
plt.colorbar(im2, ax=ax, fraction=0.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'fixture_variability_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Flag high-variability fixtures: std > 2x circuit-median on any key metric
high_var_flags = pd.DataFrame({'remote_id': fixture_var['remote_id']})
for col in available:
    high_var_flags[f'{col}_high_var'] = (fixture_var[col] > fixture_var[col].median() * 2.0).astype(int)
flag_cols = [c for c in high_var_flags.columns if c.endswith('_high_var')]
high_var_flags['n_flags'] = high_var_flags[flag_cols].sum(axis=1)
noisy = high_var_flags[high_var_flags['n_flags'] > 0].sort_values('n_flags', ascending=False)
print(f'Fixtures with elevated variability on >=1 metric: {len(noisy)}')
display(noisy.head(15))

## 13. Cross-Run Drift Analysis

Do T0001, T0002, T0003 differ systematically? High drift means the scoring notebook must use **same-run** baselines.

In [ ]:
run_means = agg_all.groupby('run_id')[available].mean().T
print('=== Circuit-level mean per run ===')
display(run_means.round(4))

run_drift_std = run_means.std(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
norm_d = run_drift_std / run_drift_std.max()
bc = ['#d62728' if v>0.5 else '#ff7f0e' if v>0.2 else '#2ca02c' for v in norm_d.values]
norm_d.plot(kind='barh', ax=ax, color=bc)
ax.axvline(0.2, color='gray', linestyle='--', lw=0.8)
ax.axvline(0.5, color='gray', linestyle=':', lw=0.8)
ax.set_title('Cross-Run Drift per Metric\nRed = use per-run baseline | Green = global acceptable')
ax.set_xlabel('Normalised std across run means')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'cross_run_drift.png', bbox_inches='tight')
plt.show()

print('Top drifting metrics (must use per-run baseline):')
print(run_drift_std.head(5).round(6).to_string())

In [ ]:
# Side-by-side cross-run comparison for four key metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
pairs = [
    ('ex_peakInputLevel_mean', 'EXCOMM - Peak Input Level'),
    ('ex_phaseNoise_dB_mean',  'EXCOMM - Phase Noise (dB)'),
    ('corr_snr_proxy',         'CORR - SNR Proxy'),
    ('rates_dsRawRate0_mean',  'RATES - DS Depth-0 Rate'),
]
for ax, (col, title) in zip(axes.flat, pairs):
    for run, color in RUN_PALETTE.items():
        sub = (agg_all[agg_all['run_id']==run].groupby('remote_id')[col].mean()
               .reset_index().sort_values('remote_id'))
        if col in sub.columns:
            ax.plot(sub['remote_id'], sub[col], color=color, label=run, linewidth=1.3)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Fixture ID')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(15))
plt.suptitle('Cross-Run Baseline Comparison (1001 -> 1090)', fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'cross_run_baseline_comparison.png', bbox_inches='tight')
plt.show()

## 14. Spatial Smoothness — Best Metrics for Change-Point Detection

In [ ]:
roughness_rows = []
for run in sorted(agg_all['run_id'].unique()):
    sub = (agg_all[agg_all['run_id']==run].groupby('remote_id')[available].mean().sort_index())
    for col in available:
        diffs = np.abs(np.diff(sub[col].values.astype(float)))
        roughness_rows.append({'run_id': run, 'metric': col,
                               'mean_abs_diff': float(np.nanmean(diffs))})

roughness_df = pd.DataFrame(roughness_rows)
roughness_pivot = roughness_df.pivot(index='metric', columns='run_id', values='mean_abs_diff')
roughness_pivot['avg_roughness'] = roughness_pivot.mean(axis=1)
roughness_pivot = roughness_pivot.sort_values('avg_roughness')

print('Spatial roughness (lower = smoother = better for change-point detection):')
display(roughness_pivot.round(6))

norm_rough = roughness_pivot['avg_roughness'] / roughness_pivot['avg_roughness'].max()
bc2 = ['#2ca02c' if v<0.25 else '#ff7f0e' if v<0.55 else '#d62728' for v in norm_rough.values]
fig, ax = plt.subplots(figsize=(10, 5))
norm_rough.plot(kind='barh', ax=ax, color=bc2)
ax.set_title('Metric Spatial Roughness under Baseline\nGreen = smooth; Red = noisy')
ax.set_xlabel('Normalised mean |delta| between adjacent fixtures')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'metric_spatial_roughness.png', bbox_inches='tight')
plt.show()

## 15. Temporal Stability within One Baseline Scenario

Is a fixture's metric stable across the full 15-minute window? Stable baselines make fault signatures more credible.

In [ ]:
ref_entry = baseline_parsed[0]
excomm_ref = ref_entry['EXCOMM'].copy()
excomm_ref = excomm_ref.dropna(subset=['remote_id','peakInputLevel','timestamp'])
excomm_ref['remote_id'] = excomm_ref['remote_id'].astype(int)

ts_min = excomm_ref['timestamp'].min()
excomm_ref['elapsed_min'] = (excomm_ref['timestamp'] - ts_min) / 60.0
excomm_ref['window'] = pd.cut(excomm_ref['elapsed_min'], bins=3,
                               labels=['0-5 min','5-10 min','10-15 min'])

w_means = (excomm_ref.groupby(['window','remote_id'])['peakInputLevel']
           .mean().reset_index().sort_values('remote_id'))

fig, ax = plt.subplots(figsize=(14, 4))
w_colors = {'0-5 min': '#1f77b4', '5-10 min': '#ff7f0e', '10-15 min': '#2ca02c'}
for window, color in w_colors.items():
    sub = w_means[w_means['window'] == window]
    ax.plot(sub['remote_id'], sub['peakInputLevel'],
            color=color, label=window, linewidth=1.3, alpha=0.9)
ax.set_title(f'EXCOMM peakInputLevel - Temporal Stability ({ref_entry["file_name"]})')
ax.set_xlabel('Fixture ID (1001 -> 1090)')
ax.set_ylabel('Peak Input Level')
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'excomm_temporal_stability.png', bbox_inches='tight')
plt.show()

## 16. Save Outputs

In [ ]:
p1 = OUTPUTS_DIR / 'baseline_fixture_features.csv'
agg_all.to_csv(p1, index=False)
print(f'Saved: {p1}  shape={agg_all.shape}')

p2 = OUTPUTS_DIR / 'baseline_stats_by_run_fixture.csv'
baseline_stats.to_csv(p2, index=False)
print(f'Saved: {p2}  shape={baseline_stats.shape}')

p3 = OUTPUTS_DIR / 'baseline_stats_global.csv'
global_stats.to_csv(p3, index=False)
print(f'Saved: {p3}  shape={global_stats.shape}')

p4 = OUTPUTS_DIR / 'fixture_variability_flags.csv'
high_var_flags.to_csv(p4, index=False)
print(f'Saved: {p4}  shape={high_var_flags.shape}')

p5 = OUTPUTS_DIR / 'metric_spatial_roughness.csv'
roughness_pivot.reset_index().to_csv(p5, index=False)
print(f'Saved: {p5}')

p6 = OUTPUTS_DIR / 'cross_run_drift.csv'
run_drift_std.reset_index().rename(columns={0: 'drift_std'}).to_csv(p6, index=False)
print(f'Saved: {p6}')

print('\nPlots:')
for f in sorted(PLOTS_DIR.glob('*.png')):
    print(f'  {f.name}')

## 17. Summary of Findings

In [ ]:
print('=' * 68)
print('BASELINE EDA - KEY FINDINGS FOR NOTEBOOKS 03-05')
print('=' * 68)

print('\n[COMM]')
rr = agg_all['comm_response_rate_mean']
print(f'  response_rate: mean={rr.mean():.2f}%, std={rr.std():.4f}% -> near-perfect under baseline')
fr = agg_all['comm_failure_rate'].dropna()
print(f'  failure_rate : mean={fr.mean():.5f}, max={fr.max():.5f} -> very low')

print('\n[EXCOMM]')
pil = agg_all['ex_peakInputLevel_mean']
print(f'  peakInputLevel: mean={pil.mean():.1f}, std={pil.std():.1f}, range=[{pil.min():.0f}, {pil.max():.0f}]')
print('  -> Clear spatial gradient along 1001->1090. Best candidate for change-point detection.')
pn = agg_all['ex_phaseNoise_dB_mean']
print(f'  phaseNoise_dB : mean={pn.mean():.3f} dB, std={pn.std():.3f}')

print('\n[CORR]')
snr = agg_all['corr_snr_proxy']
print(f'  snr_proxy: mean={snr.mean():.2f} dB, std={snr.std():.3f}, range=[{snr.min():.2f}, {snr.max():.2f}]')
print('  -> Absolute level is negative (uncorr >> corr). Change from baseline is the signal.')

print('\n[RATES]')
dr0 = agg_all['rates_dsRawRate0_mean']
print(f'  dsRawRate0: mean={dr0.mean():.4f} -> depth-0 dominates under baseline')
print('  -> Shift to higher depths in fault scenarios is a strong spatial indicator')

print('\n[SPATIAL SMOOTHNESS - top 3 smoothest]')
for m, v in roughness_pivot['avg_roughness'].nsmallest(3).items():
    print(f'  {m}: roughness={v:.6f}')

print('\n[CROSS-RUN DRIFT - top 3 drifting]')
for m, v in run_drift_std.nlargest(3).items():
    print(f'  {m}: std={v:.5f}')

print('\n[FIXTURE VARIABILITY]')
print(f'  {len(noisy)} fixtures show elevated variability on >=1 metric.')
if len(noisy): print(f'  Top noisy: {list(noisy.head(5)["remote_id"])}')

print('\n[DESIGN DECISIONS FOR NB04/05]')
print('  1. Use per-run baseline for z-score normalisation (high cross-run drift).')
print('  2. Primary families: EXCOMM (peakInputLevel) + CORR (snr_proxy).')
print('  3. Supported by RATES (depth shift) and COMM (failure rate).')
print('  4. Per-fixture z-score mandatory - fixture quirks are real and significant.')
print('  5. Spatial smoothing window: 5 fixtures (appropriate given roughness levels).')
print('=' * 68)